In [19]:
# First create the kh2lib object
from kh2lib.kh2lib import kh2lib
import os, json
lib = kh2lib()
if not os.path.isdir("workspace"):
    os.mkdir("workspace")

In [ ]:
# Now we extract the xemnas BAR into a folder in our workspace
MDLX = "pref.bin"
lib.gamedir = os.environ["KHGAMES_PATH"]
BAR_OG = os.path.join(lib.gamedir, "KH2", "bars", "root", "03system", MDLX)
print(BAR_OG)
assert os.path.isfile(BAR_OG)
BAR_OG_OUT = os.path.join(os.getcwd(), "workspace", "pref")
lib.editengine.bar_extract(BAR_OG, BAR_OG_OUT)
# C:\Users\12sam\Desktop\KH_Games\KH2\bars\root\03system

# look here https://openkh.dev/kh2/file/type/preferences.html

In [3]:
class BinaryReader:
    def __init__(self, fn=None, data=None):
        if data:
            self.data = data
        else:
            self.data = bytearray(open(fn, "rb").read())
        self.pos = 0
    def readType(self, tpe, unpack=False):
        result = {
            "float32": self.readFloat32,
            "int32": self.readInt32,
            "int16": self.readInt16
        }[tpe]()
        if unpack and tpe == "float32":
            return self.unpackFloat32(result)
        return result
    def readByte(self):
        b = self.data[self.pos]
        self.pos += 1
        return b
    def readRest(self):
        arr = bytearray()
        for _ in range(len(self.data)-self.pos):
            arr.append(self.readByte())
        return arr
    def readFloat32(self):
        import struct
        bts = bytearray([self.readByte() for _ in range(4)])
        return bts
    def unpackFloat32(self, bts):
        return struct.unpack('<f', bts)[0]
    def readInt32(self):
        bts = self.data[self.pos:self.pos+4]
        self.pos += 4
        return int.from_bytes(bts, byteorder="little")
    def readInt16(self):
        bts = self.data[self.pos:self.pos+2]
        self.pos += 2
        return int.from_bytes(bts, byteorder="little")

In [10]:
class BinaryWriter:
    def __init__(self, fn=None):
        self.fp = open(fn, "wb")
        self.nwrites = 0
    def writeType(self, tpe, data):
        return {
            "float32": self.writeFloat32,
            "int32": self.writeInt32,
            "int16": self.writeInt16
        }[tpe](data)
    def writeBytes(self, b):
        self.fp.write(b)
        self.nwrites += len(b)
        print([hex(bt) for bt in b])
    def writeArray(self, arr):
        for b in arr:
            self.writeBytes(bytes(b))
    def writeFloat32(self, value):
        packed_value = struct.pack('<f',value)
        self.writeBytes(packed_value)
    def writeInt32(self, value):
        self.writeBytes(int.to_bytes(value, length=4, byteorder="little"))
    def writeInt16(self, value):
        self.writeBytes(int.to_bytes(value, length=2, byteorder="little"))
    def close(self):
        self.fp.close()

In [5]:
import struct
#[hex(b) for b in bytearray(struct.pack('<f', BinaryFile(data=[0x64,0xD8,0x6E,0x3F]).readFloat32()))]

In [ ]:
(4) + sum([4 for _ in pointers]) + sum([sum([4 for k in ent]) for ent in entries])

In [5]:
## Read memt
import os
fn = os.path.join(os.getcwd(), "workspace", "03system", "memt.list.old")
memt = BinaryReader(fn)
file_version = memt.readInt32()
num_entries = memt.readInt32()
schema = [
    ("World ID", "int16"),
    ("World Story Flag/ID", "int16"),
    ("World Story Flag/ID Negation", "int16"),
    ("Unknown1", "int16"),
    ("Unknown2", "int16"),
    ("Unknown3", "int16"),
    ("Unknown4", "int16"),
    ("Unknown5", "int16"),
    ("Player (Sora)", "int16"),
    ("Friend 1 (Donald)", "int16"),
    ("Friend 2 (Goofy)", "int16"),
    ("World Character", "int16"),
    ("Player (Valor)", "int16"),
    ("Player (Wisdom)", "int16"),
    ("Player (Limit)", "int16"),
    ("Player (Master)", "int16"),
    ("Player (Final)", "int16"),
    ("Player (Anti)", "int16"),
    ("Player (Mickey)", "int16"),
    ("Player (Sora High Poly)", "int16"),
    ("Player (Valor High Poly)", "int16"),
    ("Player (Wisdom High Poly)", "int16"),
    ("Player (Limit High Poly)", "int16"),
    ("Player (Master High Poly)", "int16"),
    ("Player (Final High Poly)", "int16"),
    ("Player (Sora High Poly)", "int16"),
]
entries = []
for _ in range(num_entries):
    ent = {}
    for ofs in schema:
        ent[ofs[0]] = memt.readType(ofs[1], unpack=True)
    entries.append(ent)
print(memt.pos)
footer = memt.readRest()

1932


In [6]:
footer

bytearray(b'\x00\x12\x12\x12\x00\x01\x02\x12\x00\x03\x02\x01\x00C\x02\x01\x00\x83\x02\x01\x00\x03\x12\x12\x00\x01\x12\x12')

In [22]:
for ent in entries:
    ent["Friend 1 (Donald)"] = 92
    ent["World Character"] = 2078

In [23]:
# Write memt
fn = os.path.join(os.getcwd(), "workspace", "03system", "memt.list")
new = BinaryWriter(fn)
new.writeInt32(file_version)
new.writeInt32(num_entries)
for ent in entries:
    for ofs in schema:
        new.writeType(tpe=ofs[1], data=ent[ofs[0]])
new.writeBytes(footer)
new.close()
new.nwrites

['0x5', '0x0', '0x0', '0x0']
['0x25', '0x0', '0x0', '0x0']
['0x0', '0x0']
['0x0', '0x0']
['0x0', '0x0']
['0x0', '0x0']
['0x0', '0x9a']
['0x41', '0x0']
['0x0', '0xcd']
['0x18', '0x0']
['0x54', '0x0']
['0x5c', '0x0']
['0x5d', '0x0']
['0x1e', '0x8']
['0x55', '0x0']
['0x56', '0x0']
['0x5d', '0x9']
['0x57', '0x0']
['0x58', '0x0']
['0x59', '0x0']
['0x5b', '0x0']
['0x8', '0x1']
['0xc8', '0x0']
['0xf9', '0x5']
['0x7f', '0x9']
['0xfa', '0x5']
['0xfb', '0x5']
['0x8', '0x1']
['0x0', '0x0']
['0x83', '0x10']
['0x0', '0x0']
['0x0', '0x0']
['0x0', '0x0']
['0x0', '0x0']
['0x0', '0x0']
['0x0', '0x0']
['0x0', '0x0']
['0x5c', '0x0']
['0x0', '0x0']
['0x1e', '0x8']
['0x0', '0x0']
['0x0', '0x0']
['0x0', '0x0']
['0x0', '0x0']
['0x0', '0x0']
['0x0', '0x0']
['0x18', '0x3']
['0x0', '0x0']
['0x0', '0x0']
['0x0', '0x0']
['0x0', '0x0']
['0x0', '0x0']
['0x0', '0x0']
['0x0', '0x0']
['0x0', '0x0']
['0x10', '0x0']
['0x0', '0x0']
['0x0', '0x0']
['0x0', '0x0']
['0x0', '0x0']
['0x0', '0x0']
['0x0', '0x0']
['0x0', '0x0']


1960

In [24]:
# Build the 03system bar file
lib.gamedir = os.environ["KHGAMES_PATH"]
SYSTEM_OUT = os.path.join(lib.gamedir, "KH2", "KH2", "03system.bin")
SYSTEM_JSON = os.path.join(os.getcwd(), "workspace", "03system", "03system.bin.json")
lib.editengine.bar_build(SYSTEM_JSON, SYSTEM_OUT)

['pack', '-o', 'C:\\Users\\12sam\\Desktop\\KH_Games\\KH2\\KH2\\03system.bin', 'C:\\Users\\12sam\\Desktop\\git\\kh2lib\\examples\\Using kh2lib\\prefedits\\workspace\\03system\\03system.bin.json']
(None, None)


In [15]:
os.environ["KHGAMES_PATH"]

'C:\\Users\\12sam\\Desktop\\KH_Games'

In [ ]:
entries = []
for _ in range(entrycount):
    entry = {}
    entry["World ID"] = memt.readInt16()
    entry["World Flag"] = memt.readInt16()
    entry["World Flag Negation"] = memt.readInt16()
    entry["Unk1"] = memt.readInt16()
    entry["Unk2"] = memt.readInt16()
    entry["Unk3"] = memt.readInt16()
    entry["Unk4"] = memt.readInt16()
    entry["Unk5"] = memt.readInt16()
    entry["Player"] = memt.readInt16()
    entry["Friend1"] = memt.readInt16()
    entry["Friend2"] = memt.readInt16()
    entry["World Friend"] = memt.readInt16()
    entry["Valor"] = memt.readInt16()
    entry["Wisdom"] = memt.readInt16()
    entry["Limit"] = memt.readInt16()
    entry["Master"] = memt.readInt16()
    entry["Final"] = memt.readInt16()
    entry["Anti"] = memt.readInt16()
    entry["Mickey"] = memt.readInt16()
    entry["PlayerHigh"] = memt.readInt16()
    entry["ValorHigh"] = memt.readInt16()
    entry["WisdomHigh"] = memt.readInt16()
    entry["LimitHigh"] = memt.readInt16()
    entry["MasterHigh"] = memt.readInt16()
    entry["FinalHigh"] = memt.readInt16()
    entry["SoraHigh"] = memt.readInt16()
    entries.append(entry)
#     print(entry["World ID"])
    print(entry["Friend1"])
#     print(entry["World Friend"])
#     print("")

In [ ]:
MDLX = "03system.bin"
BAR = os.path.join(lib.gamedir, "KH2", "KH2",MDLX)
lib.editengine.bar_build(os.path.join(os.getcwd(), "workspace", "03system", "03system.bin.json"), BAR)